# 07 - Live AQI Inference, Route Exposure, and Health Advisory

This notebook performs inference only. It loads the trained CatBoost, LightGBM, XGBoost, scaler, selected feature list, and ensemble metadata; it does not retrain models, create datasets, or save CSV files.

Workflow: source and destination input -> coordinates -> live OpenWeather weather and pollution -> feature generation -> ensemble AQI prediction -> route waypoint AQI prediction -> cumulative exposure -> best route and health advisory.

In [1]:
# =============================================================================
# Imports, paths, and environment
# =============================================================================

import math
import os
import pickle
import warnings
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Tuple

import joblib
import numpy as np
import pandas as pd
import requests
from catboost import CatBoostRegressor
from dotenv import load_dotenv
from IPython.display import display
from xgboost import XGBRegressor

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

# Works whether the notebook is opened from project root or from Notebook/.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name.lower() == "notebook" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "Data"
PROCESSED_DIR = DATA_DIR / "Processed"
MODEL_DIR = PROJECT_ROOT / "models"

CLEANED_DATA_PATH = PROCESSED_DIR / "cleaned_data.csv"

MODEL_PATHS = {
    "CatBoost": MODEL_DIR / "catboost_model.cbm",
    "LightGBM": MODEL_DIR / "lightgbm_model.pkl",
    "XGBoost": MODEL_DIR / "xgboost_model.json",
    "Scaler": MODEL_DIR / "robust_scaler.pkl",
    "SelectedFeatures": MODEL_DIR / "selected_features.pkl",
    "EnsembleInfo": MODEL_DIR / "ensemble_info.pkl",
}

for name, path in MODEL_PATHS.items():
    if not path.exists():
        raise FileNotFoundError(f"Missing required artifact: {name} -> {path}")
if not CLEANED_DATA_PATH.exists():
    raise FileNotFoundError(f"Missing cleaned historical data: {CLEANED_DATA_PATH}")

load_dotenv(PROJECT_ROOT / ".env")
OPENWEATHER_API_KEY = os.getenv("OPENWEATHER_API_KEY")
if not OPENWEATHER_API_KEY:
    raise RuntimeError("OPENWEATHER_API_KEY was not found in .env")

OPENWEATHER_WEATHER_URL = "https://api.openweathermap.org/data/2.5/weather"
OPENWEATHER_AIR_URL = "https://api.openweathermap.org/data/2.5/air_pollution"
OPENWEATHER_GEOCODE_URL = "https://api.openweathermap.org/geo/1.0/direct"
OSRM_ROUTE_URL = "https://router.project-osrm.org/route/v1/driving"

print("Inference environment ready")
print(f"Project root: {PROJECT_ROOT}")

Inference environment ready
Project root: /Users/syedfaizaanahmad/Desktop/Project/AI-Based-Enviro-Pollution-Forecasting-Sys-for-Persona-Health-Alerts---Smart-Route-Plann-Jun-2026


## User Inputs

Set `INTERACTIVE_INPUTS = True` to type values when the cell runs. Set it to `False` and edit the variables directly for repeatable runs.

In [2]:
# =============================================================================
# User input cell
# =============================================================================

INTERACTIVE_INPUTS = True

SOURCE_CITY = ""
DESTINATION_CITY = ""
TRAVEL_DATE = datetime.now().strftime("%Y-%m-%d")
TRAVEL_TIME = datetime.now().strftime("%H:%M")

# Optional personalization for the advisory: general, sensitive, asthma, elderly, child
HEALTH_PROFILE = "general"

# API and route controls
ROUTE_POINT_COUNT = 9          # Important points sampled along each route
IDW_NEIGHBORS = 6              # Nearest historical stations used for IDW estimates
IDW_POWER = 2.0
OSRM_ROUTE_ENABLED = True      # Uses OSRM for route geometry when available; falls back to direct corridor
OSRM_MAX_ALTERNATIVES = 3
AVERAGE_SPEED_KMPH = 45.0      # Used only for fallback direct route duration
REQUIRE_LIVE_OPENWEATHER = True

if INTERACTIVE_INPUTS:
    SOURCE_CITY = input("Source City: ").strip()
    DESTINATION_CITY = input("Destination City: ").strip()
    typed_date = input(f"Travel Date [YYYY-MM-DD, default {TRAVEL_DATE}]: ").strip()
    typed_time = input(f"Travel Time [HH:MM, default {TRAVEL_TIME}]: ").strip()
    typed_profile = input("Health profile [general/sensitive/asthma/elderly/child, default general]: ").strip()

    TRAVEL_DATE = typed_date or TRAVEL_DATE
    TRAVEL_TIME = typed_time or TRAVEL_TIME
    HEALTH_PROFILE = (typed_profile or HEALTH_PROFILE).lower()

if not SOURCE_CITY or not DESTINATION_CITY:
    raise ValueError("Both SOURCE_CITY and DESTINATION_CITY are required.")

TRAVEL_DATETIME = pd.to_datetime(f"{TRAVEL_DATE} {TRAVEL_TIME}", errors="raise")
HEALTH_PROFILE = HEALTH_PROFILE.lower().strip()

print("Trip request")
print(f"Source      : {SOURCE_CITY}")
print(f"Destination : {DESTINATION_CITY}")
print(f"Travel time : {TRAVEL_DATETIME}")
print(f"Profile     : {HEALTH_PROFILE}")
print("Note: the requested date/time is used for temporal ML features. The specified OpenWeather endpoints return current live weather and air-pollution observations.")

Trip request
Source      : new delhi
Destination : patna
Travel time : 2026-07-14 20:00:00
Profile     : general
Note: the requested date/time is used for temporal ML features. The specified OpenWeather endpoints return current live weather and air-pollution observations.


## Load Trained Artifacts and Historical Reference Data

The model artifacts are loaded exactly as saved by the training pipeline. The historical cleaned data is used only for coordinates, city/state encodings, temporal AQI context, and IDW fallback estimates.

In [3]:
# =============================================================================
# Artifact loading
# =============================================================================

def load_pickle(path: Path) -> Any:
    with open(path, "rb") as f:
        return pickle.load(f)

selected_features = load_pickle(MODEL_PATHS["SelectedFeatures"])
ensemble_info = load_pickle(MODEL_PATHS["EnsembleInfo"])
scaler = load_pickle(MODEL_PATHS["Scaler"])

ensemble_features = ensemble_info.get("features", selected_features)
if list(ensemble_features) != list(selected_features):
    warnings.warn("selected_features.pkl and ensemble_info.pkl contain different feature order. Using ensemble_info['features'].")
    selected_features = list(ensemble_features)

ensemble_weights = ensemble_info.get("weights", {"CatBoost": 0.4, "LightGBM": 0.3, "XGBoost": 0.3})

catboost_model = CatBoostRegressor()
catboost_model.load_model(MODEL_PATHS["CatBoost"])

lightgbm_model = joblib.load(MODEL_PATHS["LightGBM"])

xgboost_model = XGBRegressor()
xgboost_model.load_model(MODEL_PATHS["XGBoost"])

models = {
    "CatBoost": catboost_model,
    "LightGBM": lightgbm_model,
    "XGBoost": xgboost_model,
}

print("Models and inference artifacts loaded")
print(f"Selected features: {len(selected_features)}")
print(f"Ensemble weights : {ensemble_weights}")

Models and inference artifacts loaded
Selected features: 47
Ensemble weights : {'CatBoost': 0.4, 'LightGBM': 0.3, 'XGBoost': 0.3}


In [4]:
# =============================================================================
# Historical data load for coordinates and IDW context
# =============================================================================

BASE_COLUMNS = [
    "timestamp", "state", "city", "latitude", "longitude", "calculated_aqi",
    "pm25", "pm10", "nitric_oxide", "nitrogen_dioxide", "nitrogen_oxides",
    "ammonia", "sulfur_dioxide", "carbon_monoxide", "ozone",
    "ambient_temperature", "relative_humidity", "solar_radiation", "rainfall",
    "month", "hour", "dayofweek", "is_weekend", "state_encoded", "city_encoded",
]

available_columns = pd.read_csv(CLEANED_DATA_PATH, nrows=0).columns.tolist()
historical_usecols = [col for col in BASE_COLUMNS if col in available_columns]

historical_df = pd.read_csv(CLEANED_DATA_PATH, usecols=historical_usecols, parse_dates=["timestamp"])
historical_df = historical_df.sort_values(["city", "timestamp"]).reset_index(drop=True)

print("Historical reference data loaded")
print(f"Rows     : {historical_df.shape[0]:,}")
print(f"Stations : {historical_df[['state', 'city', 'latitude', 'longitude']].drop_duplicates().shape[0]:,}")
print(f"Date span: {historical_df['timestamp'].min()} to {historical_df['timestamp'].max()}")

Historical reference data loaded
Rows     : 3,429,120
Stations : 66
Date span: 2017-01-01 00:00:00 to 2026-06-30 23:00:00


## Feature Engineering Helpers

These functions mirror the training feature names and scaling contract. Raw features are created first, missing live values are filled from historical/IDW context, and `robust_scaler.pkl` is applied only after the feature row is complete and ordered.

In [5]:
# =============================================================================
# General utilities
# =============================================================================

EPSILON = 1e-6
API_CACHE: Dict[Tuple[str, Tuple[Tuple[str, Any], ...]], Dict[str, Any]] = {}


def normalize_name(value: Any) -> str:
    text = "" if pd.isna(value) else str(value).lower()
    for char in "(),-_./\\":
        text = text.replace(char, " ")
    return " ".join(text.split())


def safe_float(value: Any, default: float = np.nan) -> float:
    try:
        if value is None:
            return default
        out = float(value)
        if math.isnan(out) or math.isinf(out):
            return default
        return out
    except (TypeError, ValueError):
        return default


def safe_divide(numerator: Any, denominator: Any) -> float:
    numerator = safe_float(numerator, 0.0)
    denominator = safe_float(denominator, 0.0)
    return numerator / (denominator + EPSILON)


def haversine_km(lat1: float, lon1: float, lat2: float, lon2: float) -> float:
    radius_km = 6371.0088
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    d_phi = math.radians(lat2 - lat1)
    d_lambda = math.radians(lon2 - lon1)
    a = math.sin(d_phi / 2) ** 2 + math.cos(phi1) * math.cos(phi2) * math.sin(d_lambda / 2) ** 2
    return 2 * radius_km * math.atan2(math.sqrt(a), math.sqrt(1 - a))


def add_temporal_features(features: Dict[str, Any], when: pd.Timestamp) -> Dict[str, Any]:
    when = pd.to_datetime(when)
    month = int(when.month)
    hour = int(when.hour)
    dayofweek = int(when.dayofweek)

    features["Year"] = int(when.year)
    features["month"] = month
    features["hour"] = hour
    features["dayofweek"] = dayofweek
    features["is_weekend"] = int(dayofweek >= 5)
    features["Day"] = int(when.day)

    features["Month_sin"] = math.sin(2 * math.pi * month / 12)
    features["Month_cos"] = math.cos(2 * math.pi * month / 12)
    features["DayOfWeek_sin"] = math.sin(2 * math.pi * dayofweek / 7)
    features["DayOfWeek_cos"] = math.cos(2 * math.pi * dayofweek / 7)
    features["Hour_sin"] = math.sin(2 * math.pi * hour / 24)
    features["Hour_cos"] = math.cos(2 * math.pi * hour / 24)
    return features


def add_interaction_features(features: Dict[str, Any]) -> Dict[str, Any]:
    features["PM25_PM10_Ratio"] = safe_divide(features.get("pm25"), features.get("pm10"))
    features["NO2_SO2_Ratio"] = safe_divide(features.get("nitrogen_dioxide"), features.get("sulfur_dioxide"))
    features["CO_NO2_Ratio"] = safe_divide(features.get("carbon_monoxide"), features.get("nitrogen_dioxide"))
    features["O3_NO2_Ratio"] = safe_divide(features.get("ozone"), features.get("nitrogen_dioxide"))

    features["Temp_Humidity"] = safe_float(features.get("ambient_temperature"), 0.0) * safe_float(features.get("relative_humidity"), 0.0)
    features["Humidity_Solar"] = safe_float(features.get("relative_humidity"), 0.0) * safe_float(features.get("solar_radiation"), 0.0)

    features["pm25_ambient_temperature"] = safe_float(features.get("pm25"), 0.0) * safe_float(features.get("ambient_temperature"), 0.0)
    features["pm25_relative_humidity"] = safe_float(features.get("pm25"), 0.0) * safe_float(features.get("relative_humidity"), 0.0)
    features["pm10_relative_humidity"] = safe_float(features.get("pm10"), 0.0) * safe_float(features.get("relative_humidity"), 0.0)
    features["nitrogen_dioxide_solar_radiation"] = safe_float(features.get("nitrogen_dioxide"), 0.0) * safe_float(features.get("solar_radiation"), 0.0)
    features["sulfur_dioxide_ambient_temperature"] = safe_float(features.get("sulfur_dioxide"), 0.0) * safe_float(features.get("ambient_temperature"), 0.0)
    return features


def clean_feature_frame(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for col in selected_features:
        if col not in df.columns:
            df[col] = np.nan
    df = df[selected_features].apply(pd.to_numeric, errors="coerce")
    df = df.replace([np.inf, -np.inf], np.nan)
    for col in selected_features:
        default = RAW_FEATURE_DEFAULTS.get(col, 0.0)
        df[col] = df[col].fillna(default)
    return df.fillna(0.0)

In [6]:
# =============================================================================
# Build station-level profiles for IDW and historical fallbacks
# =============================================================================

BASE_NUMERIC_COLUMNS = [
    "pm25", "pm10", "nitric_oxide", "nitrogen_dioxide", "nitrogen_oxides",
    "ammonia", "sulfur_dioxide", "carbon_monoxide", "ozone",
    "ambient_temperature", "relative_humidity", "solar_radiation", "rainfall",
    "state_encoded", "city_encoded",
]


def lag_value(series: pd.Series, lag: int) -> float:
    series = series.dropna()
    if series.empty:
        return np.nan
    if len(series) > lag:
        return safe_float(series.iloc[-lag - 1])
    return safe_float(series.iloc[0])


def rolling_std_value(series: pd.Series, window: int) -> float:
    series = series.dropna().tail(window)
    if len(series) <= 1:
        return 0.0
    return safe_float(series.std(ddof=1), 0.0)


def build_station_profiles(df: pd.DataFrame) -> pd.DataFrame:
    profile_rows: List[Dict[str, Any]] = []
    group_cols = ["state", "city", "latitude", "longitude"]

    for station_id, (keys, group) in enumerate(df.groupby(group_cols, dropna=False, sort=False)):
        state, city, latitude, longitude = keys
        group = group.sort_values("timestamp")
        latest = group.iloc[-1]
        target = group["calculated_aqi"].astype(float)

        row: Dict[str, Any] = {
            "station_id": station_id,
            "state": state,
            "city": city,
            "station_label": f"{city}, {state}",
            "latitude": safe_float(latitude),
            "longitude": safe_float(longitude),
            "latest_timestamp": latest["timestamp"],
            "calculated_aqi": safe_float(latest.get("calculated_aqi")),
            "calculated_aqi_lag_1": safe_float(target.iloc[-1]) if len(target) else np.nan,
            "calculated_aqi_lag_24": lag_value(target, 24),
            "calculated_aqi_lag_48": lag_value(target, 48),
            "calculated_aqi_lag_72": lag_value(target, 72),
            "calculated_aqi_rolling_std_3": rolling_std_value(target, 3),
            "calculated_aqi_rolling_std_6": rolling_std_value(target, 6),
            "calculated_aqi_rolling_std_12": rolling_std_value(target, 12),
            "calculated_aqi_rolling_std_24": rolling_std_value(target, 24),
            "calculated_aqi_expanding_mean": safe_float(target.mean()),
        }

        for col in BASE_NUMERIC_COLUMNS:
            if col in latest.index:
                row[col] = safe_float(latest[col])

        add_temporal_features(row, latest["timestamp"])
        add_interaction_features(row)
        profile_rows.append(row)

    profiles = pd.DataFrame(profile_rows)
    profiles["city_norm"] = profiles["city"].map(normalize_name)
    profiles["state_norm"] = profiles["state"].map(normalize_name)

    for feature in selected_features:
        if feature not in profiles.columns:
            profiles[feature] = np.nan

    return profiles


station_profiles = build_station_profiles(historical_df)
RAW_FEATURE_DEFAULTS = (
    station_profiles[selected_features]
    .apply(pd.to_numeric, errors="coerce")
    .replace([np.inf, -np.inf], np.nan)
    .median(numeric_only=True)
    .fillna(0.0)
    .to_dict()
)

print("Station profiles ready for historical lookup and IDW")
display(station_profiles[["station_id", "state", "city", "latitude", "longitude", "latest_timestamp"]].head())

Station profiles ready for historical lookup and IDW


,station_id,state,city,latitude,longitude,latest_timestamp
0,0,Chhattisgarh,"32Bungalows, Bhilai (, )",21.194815,81.31477,2025-12-31 23:00:00
1,1,Chhattisgarh,"AIIMS, Raipur (, )",21.256472,81.57914,2025-12-31 23:00:00
2,2,Delhi,"Alipur (, )",28.797900,77.12330,2026-06-30 23:00:00
3,3,Andhra Pradesh,"Anand Kala Kshetram, Rajamahendravaram (, )",16.987288,81.73632,2026-06-30 23:00:00
4,4,Delhi,"Anand Vihar ( , )",28.650800,77.31520,2026-06-30 23:00:00


## Live API and Location Resolution

City names are resolved from `cleaned_data.csv` first. If no historical match is found, OpenWeather Geocoding is used. Pollutant/weather values come from OpenWeather for every predicted coordinate, with IDW used for missing values and historical context features.

In [7]:
# =============================================================================
# OpenWeather API helpers
# =============================================================================


def cacheable_params(params: Dict[str, Any]) -> Tuple[Tuple[str, Any], ...]:
    items = []
    for key, value in sorted(params.items()):
        if key == "appid":
            continue
        if isinstance(value, float):
            value = round(value, 5)
        items.append((key, value))
    return tuple(items)


def openweather_get(url: str, params: Dict[str, Any], timeout: int = 25) -> Dict[str, Any]:
    request_params = dict(params)
    request_params["appid"] = OPENWEATHER_API_KEY
    key = (url, cacheable_params(request_params))
    if key in API_CACHE:
        return API_CACHE[key]

    response = requests.get(url, params=request_params, timeout=timeout)
    if response.status_code == 401:
        raise RuntimeError("OpenWeather rejected the API key. Check OPENWEATHER_API_KEY in .env.")
    response.raise_for_status()
    data = response.json()
    API_CACHE[key] = data
    return data


def geocode_city(query: str) -> Dict[str, Any]:
    data = openweather_get(OPENWEATHER_GEOCODE_URL, {"q": query, "limit": 1})
    if not data:
        raise ValueError(f"OpenWeather Geocoding could not resolve city: {query}")
    item = data[0]
    return {
        "query": query,
        "display_name": ", ".join(part for part in [item.get("name"), item.get("state"), item.get("country")] if part),
        "latitude": safe_float(item.get("lat")),
        "longitude": safe_float(item.get("lon")),
        "state": item.get("state"),
        "country": item.get("country"),
        "source": "openweather_geocoding",
        "match_type": "geocoded",
        "station_ids": [],
    }


def fetch_current_weather(latitude: float, longitude: float) -> Tuple[Dict[str, float], Dict[str, Any]]:
    data = openweather_get(
        OPENWEATHER_WEATHER_URL,
        {"lat": latitude, "lon": longitude, "units": "metric"},
    )
    main = data.get("main", {}) or {}
    rain = data.get("rain", {}) or {}
    weather_items = data.get("weather", []) or [{}]

    values = {
        "ambient_temperature": safe_float(main.get("temp")),
        "relative_humidity": safe_float(main.get("humidity")),
        "rainfall": safe_float(rain.get("1h", rain.get("3h", 0.0)), 0.0),
    }
    meta = {
        "weather_description": weather_items[0].get("description"),
        "weather_station_name": data.get("name"),
    }
    return values, meta


def fetch_current_air_pollution(latitude: float, longitude: float) -> Tuple[Dict[str, float], Dict[str, Any]]:
    data = openweather_get(OPENWEATHER_AIR_URL, {"lat": latitude, "lon": longitude})
    records = data.get("list", [])
    if not records:
        raise ValueError("OpenWeather Air Pollution API returned no records for the coordinate.")

    record = records[0]
    components = record.get("components", {}) or {}
    values = {
        "pm25": safe_float(components.get("pm2_5")),
        "pm10": safe_float(components.get("pm10")),
        "nitric_oxide": safe_float(components.get("no")),
        "nitrogen_dioxide": safe_float(components.get("no2")),
        "ammonia": safe_float(components.get("nh3")),
        "sulfur_dioxide": safe_float(components.get("so2")),
        "ozone": safe_float(components.get("o3")),
    }

    # OpenWeather reports CO in micrograms/m3. The historical feature scale is mg/m3-like.
    if components.get("co") is not None:
        values["carbon_monoxide"] = safe_float(components.get("co")) / 1000.0

    if not np.isnan(values.get("nitric_oxide", np.nan)) and not np.isnan(values.get("nitrogen_dioxide", np.nan)):
        values["nitrogen_oxides"] = values["nitric_oxide"] + values["nitrogen_dioxide"]

    meta = {
        "openweather_aqi_index": (record.get("main", {}) or {}).get("aqi"),
        "air_pollution_timestamp": pd.to_datetime(record.get("dt"), unit="s") if record.get("dt") else None,
    }
    return values, meta


def maybe_fetch_live_values(latitude: float, longitude: float) -> Tuple[Dict[str, Any], Dict[str, Any]]:
    values: Dict[str, Any] = {}
    meta: Dict[str, Any] = {}

    try:
        weather_values, weather_meta = fetch_current_weather(latitude, longitude)
        values.update({k: v for k, v in weather_values.items() if not np.isnan(v)})
        meta.update(weather_meta)
    except Exception as exc:
        if REQUIRE_LIVE_OPENWEATHER:
            raise
        warnings.warn(f"Weather API failed; using historical/IDW fallback. Error: {exc}")

    try:
        pollution_values, pollution_meta = fetch_current_air_pollution(latitude, longitude)
        values.update({k: v for k, v in pollution_values.items() if not np.isnan(v)})
        meta.update(pollution_meta)
    except Exception as exc:
        if REQUIRE_LIVE_OPENWEATHER:
            raise
        warnings.warn(f"Air Pollution API failed; using historical/IDW fallback. Error: {exc}")

    return values, meta

In [8]:
# =============================================================================
# Historical city lookup and IDW profile estimation
# =============================================================================


def find_historical_location(query: str) -> Optional[Dict[str, Any]]:
    query_norm = normalize_name(query)
    if not query_norm:
        return None

    match_type = None
    candidates = station_profiles[station_profiles["city_norm"] == query_norm]
    if len(candidates):
        match_type = "exact_city"
    else:
        candidates = station_profiles[station_profiles["state_norm"] == query_norm]
        if len(candidates):
            match_type = "state_match"
        else:
            candidates = station_profiles[station_profiles["city_norm"].str.contains(query_norm, regex=False, na=False)]
            if len(candidates):
                match_type = "city_contains_query"

    if not len(candidates):
        return None

    latitude = safe_float(candidates["latitude"].median())
    longitude = safe_float(candidates["longitude"].median())
    state = candidates["state"].mode().iloc[0] if len(candidates["state"].mode()) else candidates.iloc[0]["state"]
    city_label = candidates["city"].mode().iloc[0] if len(candidates["city"].mode()) else candidates.iloc[0]["city"]

    return {
        "query": query,
        "display_name": f"{query} (historical match: {city_label}, {state})",
        "latitude": latitude,
        "longitude": longitude,
        "state": state,
        "country": None,
        "source": "cleaned_data.csv",
        "match_type": match_type,
        "station_ids": candidates["station_id"].tolist(),
        "matched_station_count": int(len(candidates)),
    }


def resolve_city(query: str) -> Dict[str, Any]:
    historical_match = find_historical_location(query)
    if historical_match is not None:
        return historical_match
    return geocode_city(query)


def idw_station_profile(latitude: float, longitude: float, station_ids: Optional[Iterable[int]] = None, k: int = IDW_NEIGHBORS) -> Tuple[Dict[str, float], pd.DataFrame]:
    if station_ids:
        candidates = station_profiles[station_profiles["station_id"].isin(list(station_ids))].copy()
    else:
        candidates = station_profiles.copy()

    if candidates.empty:
        raise ValueError("No station profiles are available for IDW estimation.")

    candidates["distance_km"] = candidates.apply(
        lambda row: haversine_km(latitude, longitude, row["latitude"], row["longitude"]),
        axis=1,
    )
    nearest = candidates.nsmallest(max(1, min(k, len(candidates))), "distance_km").copy()

    if nearest["distance_km"].min() < 0.01:
        exact_row = nearest.iloc[0]
        weights = np.zeros(len(nearest), dtype=float)
        weights[0] = 1.0
    else:
        inv_dist = 1.0 / np.power(nearest["distance_km"].to_numpy(dtype=float) + EPSILON, IDW_POWER)
        weights = inv_dist / inv_dist.sum()

    profile: Dict[str, float] = {}
    numeric_columns = [col for col in selected_features if col in nearest.columns]
    for col in numeric_columns:
        values = pd.to_numeric(nearest[col], errors="coerce").to_numpy(dtype=float)
        valid = ~np.isnan(values)
        if valid.any():
            local_weights = weights[valid]
            local_weights = local_weights / local_weights.sum()
            profile[col] = float(np.dot(values[valid], local_weights))

    nearest["idw_weight"] = weights
    return profile, nearest[["station_id", "state", "city", "latitude", "longitude", "distance_km", "idw_weight"]]


def feature_row_for_coordinate(
    latitude: float,
    longitude: float,
    when: pd.Timestamp,
    station_ids: Optional[Iterable[int]] = None,
    label: str = "location",
) -> Tuple[Dict[str, Any], Dict[str, Any]]:
    idw_profile, nearest = idw_station_profile(latitude, longitude, station_ids=station_ids)

    raw_features = dict(RAW_FEATURE_DEFAULTS)
    raw_features.update(idw_profile)

    live_values, live_meta = maybe_fetch_live_values(latitude, longitude)
    raw_features.update(live_values)

    # Current weather API does not provide solar radiation; retain historical/IDW estimate.
    if "solar_radiation" not in raw_features or pd.isna(raw_features.get("solar_radiation")):
        raw_features["solar_radiation"] = RAW_FEATURE_DEFAULTS.get("solar_radiation", 0.0)

    add_temporal_features(raw_features, when)
    add_interaction_features(raw_features)

    for feature in selected_features:
        if feature not in raw_features or pd.isna(raw_features[feature]):
            raw_features[feature] = RAW_FEATURE_DEFAULTS.get(feature, 0.0)

    meta = {
        "label": label,
        "latitude": latitude,
        "longitude": longitude,
        "nearest_stations": nearest,
        **live_meta,
    }
    return raw_features, meta


def predict_aqi_from_raw(raw_features: Dict[str, Any]) -> Dict[str, float]:
    x_raw = clean_feature_frame(pd.DataFrame([raw_features]))
    x_scaled = pd.DataFrame(scaler.transform(x_raw), columns=selected_features)

    predictions = {
        "CatBoost": float(models["CatBoost"].predict(x_scaled)[0]),
        "LightGBM": float(models["LightGBM"].predict(x_scaled)[0]),
        "XGBoost": float(models["XGBoost"].predict(x_scaled)[0]),
    }
    ensemble_prediction = sum(float(ensemble_weights.get(name, 0.0)) * value for name, value in predictions.items())
    predictions["Ensemble"] = max(0.0, float(ensemble_prediction))
    return predictions

## Route, Exposure, and Advisory Helpers

OSRM is used only for route geometry when available. If the route service is unavailable, the notebook falls back to a direct source-to-destination corridor so AQI exposure can still be estimated from live OpenWeather observations and IDW context.

In [9]:
# =============================================================================
# Route helpers
# =============================================================================


def route_distance_km(points: List[Tuple[float, float]]) -> float:
    if len(points) < 2:
        return 0.0
    return sum(haversine_km(points[i][0], points[i][1], points[i + 1][0], points[i + 1][1]) for i in range(len(points) - 1))


def sample_points_by_distance(points: List[Tuple[float, float]], n_points: int) -> List[Tuple[float, float]]:
    if not points:
        return []
    if len(points) == 1 or n_points <= 1:
        return [points[0]]

    segment_lengths = [0.0]
    for i in range(len(points) - 1):
        segment_lengths.append(haversine_km(points[i][0], points[i][1], points[i + 1][0], points[i + 1][1]))
    cumulative = np.cumsum(segment_lengths)
    total = cumulative[-1]

    if total <= 0:
        return [points[0] for _ in range(n_points)]

    sampled: List[Tuple[float, float]] = []
    targets = np.linspace(0, total, n_points)
    seg_idx = 1
    for target in targets:
        while seg_idx < len(cumulative) - 1 and cumulative[seg_idx] < target:
            seg_idx += 1
        start_dist = cumulative[seg_idx - 1]
        end_dist = cumulative[seg_idx]
        start = points[seg_idx - 1]
        end = points[seg_idx]
        ratio = 0.0 if end_dist == start_dist else (target - start_dist) / (end_dist - start_dist)
        lat = start[0] + ratio * (end[0] - start[0])
        lon = start[1] + ratio * (end[1] - start[1])
        sampled.append((float(lat), float(lon)))

    return sampled


def fetch_osrm_routes(source: Dict[str, Any], destination: Dict[str, Any]) -> List[Dict[str, Any]]:
    if not OSRM_ROUTE_ENABLED:
        return []

    coordinates = f"{source['longitude']},{source['latitude']};{destination['longitude']},{destination['latitude']}"
    params = {
        "overview": "full",
        "geometries": "geojson",
        "alternatives": "true",
        "steps": "false",
    }

    try:
        response = requests.get(f"{OSRM_ROUTE_URL}/{coordinates}", params=params, timeout=25)
        response.raise_for_status()
        data = response.json()
    except Exception as exc:
        warnings.warn(f"OSRM route lookup failed; using direct fallback route. Error: {exc}")
        return []

    routes = []
    for idx, route in enumerate(data.get("routes", [])[:OSRM_MAX_ALTERNATIVES], start=1):
        coords = route.get("geometry", {}).get("coordinates", [])
        points = [(safe_float(lat), safe_float(lon)) for lon, lat in coords]
        if len(points) < 2:
            continue
        sampled = sample_points_by_distance(points, ROUTE_POINT_COUNT)
        routes.append({
            "route_name": f"OSRM route {idx}",
            "route_source": "OSRM",
            "points": sampled,
            "geometry_points": points,
            "distance_km": safe_float(route.get("distance"), 0.0) / 1000.0,
            "duration_hours": safe_float(route.get("duration"), 0.0) / 3600.0,
        })
    return routes


def build_direct_route(source: Dict[str, Any], destination: Dict[str, Any]) -> Dict[str, Any]:
    points = []
    for frac in np.linspace(0, 1, ROUTE_POINT_COUNT):
        lat = source["latitude"] + frac * (destination["latitude"] - source["latitude"])
        lon = source["longitude"] + frac * (destination["longitude"] - source["longitude"])
        points.append((float(lat), float(lon)))
    distance = route_distance_km(points)
    return {
        "route_name": "Direct corridor fallback",
        "route_source": "geodesic_interpolation",
        "points": points,
        "geometry_points": points,
        "distance_km": distance,
        "duration_hours": distance / AVERAGE_SPEED_KMPH if AVERAGE_SPEED_KMPH > 0 else 0.0,
    }


def build_candidate_routes(source: Dict[str, Any], destination: Dict[str, Any]) -> List[Dict[str, Any]]:
    routes = fetch_osrm_routes(source, destination)
    if routes:
        return routes
    return [build_direct_route(source, destination)]

In [10]:
# =============================================================================
# AQI categories, route prediction, exposure, and health advisory
# =============================================================================


def aqi_category(aqi: float) -> str:
    aqi = safe_float(aqi, 0.0)
    if aqi <= 50:
        return "Good"
    if aqi <= 100:
        return "Satisfactory"
    if aqi <= 200:
        return "Moderate"
    if aqi <= 300:
        return "Poor"
    if aqi <= 400:
        return "Very Poor"
    return "Severe"


def predict_route(route: Dict[str, Any], source: Dict[str, Any], destination: Dict[str, Any], start_time: pd.Timestamp) -> Dict[str, Any]:
    rows: List[Dict[str, Any]] = []
    points = route["points"]
    n_points = len(points)
    duration_hours = max(float(route.get("duration_hours", 0.0)), 0.0)

    for idx, (lat, lon) in enumerate(points):
        if idx == 0:
            label = f"Source: {source['query']}"
            station_ids = source.get("station_ids")
        elif idx == n_points - 1:
            label = f"Destination: {destination['query']}"
            station_ids = destination.get("station_ids")
        else:
            label = f"Waypoint {idx}"
            station_ids = None

        point_time = start_time + pd.Timedelta(hours=(duration_hours * idx / max(n_points - 1, 1)))
        raw_features, meta = feature_row_for_coordinate(lat, lon, point_time, station_ids=station_ids, label=label)
        preds = predict_aqi_from_raw(raw_features)
        nearest = meta["nearest_stations"].iloc[0]

        rows.append({
            "route": route["route_name"],
            "location": label,
            "eta": point_time,
            "latitude": lat,
            "longitude": lon,
            "predicted_aqi": preds["Ensemble"],
            "aqi_category": aqi_category(preds["Ensemble"]),
            "catboost_aqi": preds["CatBoost"],
            "lightgbm_aqi": preds["LightGBM"],
            "xgboost_aqi": preds["XGBoost"],
            "pm25": raw_features.get("pm25"),
            "pm10": raw_features.get("pm10"),
            "temperature_c": raw_features.get("ambient_temperature"),
            "humidity_pct": raw_features.get("relative_humidity"),
            "weather": meta.get("weather_description"),
            "openweather_aqi_index": meta.get("openweather_aqi_index"),
            "nearest_historical_station": nearest["city"],
            "nearest_station_distance_km": nearest["distance_km"],
        })

    points_df = pd.DataFrame(rows)
    aqi_values = points_df["predicted_aqi"].to_numpy(dtype=float)
    if len(aqi_values) > 1 and duration_hours > 0:
        segment_hours = duration_hours / (len(aqi_values) - 1)
        exposure = float(sum(((aqi_values[i] + aqi_values[i + 1]) / 2.0) * segment_hours for i in range(len(aqi_values) - 1)))
    else:
        exposure = float(aqi_values.mean()) if len(aqi_values) else 0.0

    summary = {
        "route": route["route_name"],
        "route_source": route["route_source"],
        "distance_km": float(route.get("distance_km", 0.0)),
        "duration_hours": duration_hours,
        "average_aqi": float(points_df["predicted_aqi"].mean()),
        "max_aqi": float(points_df["predicted_aqi"].max()),
        "worst_category": aqi_category(float(points_df["predicted_aqi"].max())),
        "exposure_aqi_hours": exposure,
    }
    return {"summary": summary, "points": points_df}


def build_health_advisory(summary: Dict[str, Any], health_profile: str) -> str:
    max_aqi = summary["max_aqi"]
    avg_aqi = summary["average_aqi"]
    category = summary["worst_category"]
    exposure = summary["exposure_aqi_hours"]
    profile = health_profile.lower().strip()
    sensitive = profile in {"sensitive", "asthma", "elderly", "child"}

    lines = [
        f"Recommended route: {summary['route']}",
        f"Expected exposure: {exposure:.1f} AQI-hours over {summary['duration_hours']:.2f} hours.",
        f"Average AQI: {avg_aqi:.1f}; peak AQI: {max_aqi:.1f} ({category}).",
    ]

    if category in {"Good", "Satisfactory"}:
        lines.append("Air quality is acceptable for most travelers. Keep normal hydration and ventilation practices.")
    elif category == "Moderate":
        lines.append("Consider closing windows in dense traffic and using cabin recirculation during polluted stretches.")
        if sensitive:
            lines.append("Because the profile is sensitive, carry prescribed medication and reduce outdoor stops.")
    elif category == "Poor":
        lines.append("Limit outdoor exposure at waypoints, use cabin recirculation, and avoid heavy exertion during breaks.")
        if sensitive:
            lines.append("Sensitive travelers should consider an N95 mask during outdoor transfers and keep medication accessible.")
    else:
        lines.append("High pollution risk. Prefer the lowest-exposure route, minimize stops, keep windows closed, and use filtered cabin air.")
        if sensitive:
            lines.append("Sensitive travelers should consider delaying travel or choosing a cleaner travel window if practical.")

    return "\n".join(lines)

## Run Live Inference

Run this cell after the setup cells. It resolves both cities, builds candidate route(s), predicts AQI at each important route point, computes cumulative pollution exposure, and prints the route recommendation plus health advisory.

In [11]:
# =============================================================================
# End-to-end live inference run
# =============================================================================

source_location = resolve_city(SOURCE_CITY)
destination_location = resolve_city(DESTINATION_CITY)

print("Resolved locations")
display(pd.DataFrame([source_location, destination_location])[[
    "query", "display_name", "latitude", "longitude", "source", "match_type"
]])

candidate_routes = build_candidate_routes(source_location, destination_location)
print(f"Candidate routes found: {len(candidate_routes)}")

route_results = []
for route in candidate_routes:
    print(f"Predicting route exposure: {route['route_name']}")
    route_results.append(predict_route(route, source_location, destination_location, TRAVEL_DATETIME))

summary_df = pd.DataFrame([result["summary"] for result in route_results]).sort_values("exposure_aqi_hours").reset_index(drop=True)
best_route_name = summary_df.iloc[0]["route"]
best_result = next(result for result in route_results if result["summary"]["route"] == best_route_name)

print("Route exposure summary")
display(summary_df)

print("Best route waypoint predictions")
waypoint_columns = [
    "location", "eta", "latitude", "longitude", "predicted_aqi", "aqi_category",
    "pm25", "pm10", "temperature_c", "humidity_pct", "weather",
    "openweather_aqi_index", "nearest_historical_station", "nearest_station_distance_km",
]
display(best_result["points"][waypoint_columns])

print("Health advisory")
print(build_health_advisory(best_result["summary"], HEALTH_PROFILE))

Resolved locations


,query,display_name,latitude,longitude,source,match_type
0,new delhi,"New Delhi, Delhi, IN",28.613895,77.209006,openweather_geocoding,geocoded
1,patna,"patna (historical match: GVM Corporation, Visa...",17.720000,83.300000,cleaned_data.csv,city_contains_query


Candidate routes found: 2
Predicting route exposure: OSRM route 1
Predicting route exposure: OSRM route 2
Route exposure summary


,route,route_source,distance_km,duration_hours,average_aqi,max_aqi,worst_category,exposure_aqi_hours
0,OSRM route 1,OSRM,1707.5062,21.126583,96.498649,247.498293,Poor,1817.712850
1,OSRM route 2,OSRM,1799.5569,22.375083,95.152334,247.498293,Poor,1891.465659


Best route waypoint predictions


,location,eta,latitude,longitude,predicted_aqi,aqi_category,pm25,pm10,temperature_c,humidity_pct,weather,openweather_aqi_index,nearest_historical_station,nearest_station_distance_km
0,Source: new delhi,2026-07-14 20:00:00.000000000,28.613909,77.209007,247.498293,Poor,17.05,54.03,39.20,25.0,few clouds,3,"Ashok Vihar (, )",9.445032
1,Waypoint 1,2026-07-14 22:38:26.962500000,26.987871,77.835546,144.384354,Moderate,16.70,48.15,38.47,33.0,overcast clouds,2,"Manoharpur, Agra (, )",33.007946
2,Waypoint 2,2026-07-15 01:16:53.925000000,25.500788,78.546145,34.576145,Good,9.46,15.81,38.36,27.0,overcast clouds,2,"Shivaji Nagar, Jhansi (, )",7.383897
3,Waypoint 3,2026-07-15 03:55:20.887500000,24.073499,79.366532,31.770220,Good,9.96,12.65,32.46,48.0,overcast clouds,2,"Shrivastav Colony, Damoh (, )",29.364943
4,Waypoint 4,2026-07-15 06:33:47.850000000,22.744942,80.303003,56.361839,Satisfactory,11.29,14.34,28.98,63.0,overcast clouds,2,"Gole Bazar, Katni (, )",121.407564
5,Waypoint 5,2026-07-15 09:12:14.812500000,21.691929,81.550059,77.527238,Satisfactory,12.22,14.76,32.96,48.0,broken clouds,2,"AIIMS, Raipur (, )",48.514047
6,Waypoint 6,2026-07-15 11:50:41.775000000,20.204606,82.185998,80.233789,Satisfactory,16.29,20.30,25.87,76.0,overcast clouds,2,"AIIMS, Raipur (, )",132.902160
7,Waypoint 7,2026-07-15 14:29:08.737499999,18.843016,82.604192,83.284654,Satisfactory,14.25,20.65,27.16,72.0,overcast clouds,2,"GVM Corporation, Visakhapatnam (, )",144.880556
8,Destination: patna,2026-07-15 17:07:35.700000000,17.720019,83.300000,112.851308,Moderate,20.11,31.10,25.08,84.0,overcast clouds,2,"GVM Corporation, Visakhapatnam (, )",0.002113


Health advisory
Recommended route: OSRM route 1
Expected exposure: 1817.7 AQI-hours over 21.13 hours.
Average AQI: 96.5; peak AQI: 247.5 (Poor).
Limit outdoor exposure at waypoints, use cabin recirculation, and avoid heavy exertion during breaks.


## Additional Live Reports

Run these cells after the end-to-end inference cell. They reuse the resolved locations, selected route, live API cache, and trained ensemble.


In [12]:
# =============================================================================
# AQI report for source, destination, and major monitored cities along the route
# =============================================================================

ROUTE_CITY_CORRIDOR_KM = 80.0
MIN_ROUTE_CITY_DISTANCE_FROM_ENDPOINT_KM = 20.0
MAX_MAJOR_ROUTE_CITIES = None  # None means all matching historical monitoring locations on the route corridor.

required_objects = ["source_location", "destination_location", "candidate_routes", "best_result", "best_route_name"]
missing_objects = [name for name in required_objects if name not in globals()]
if missing_objects:
    raise RuntimeError(f"Run the end-to-end live inference cell first. Missing: {missing_objects}")


def readable_monitoring_city(city_value: Any, state_value: Any) -> str:
    text = str(city_value).split("(")[0].strip(" ,")
    parts = [part.strip() for part in text.split(",") if part.strip()]
    if len(parts) >= 2:
        return f"{parts[-1]} ({parts[0]}, {state_value})"
    return f"{text}, {state_value}"


def nearest_distance_to_route_km(latitude: float, longitude: float, route_points: List[Tuple[float, float]]) -> float:
    if not route_points:
        return np.inf
    return min(haversine_km(latitude, longitude, point_lat, point_lon) for point_lat, point_lon in route_points)


def route_progress_fraction(latitude: float, longitude: float, route_points: List[Tuple[float, float]]) -> float:
    if len(route_points) <= 1:
        return 0.0
    distances = [haversine_km(latitude, longitude, point_lat, point_lon) for point_lat, point_lon in route_points]
    return float(int(np.argmin(distances)) / max(len(route_points) - 1, 1))


def route_aqi_report_row(label: str, role: str, latitude: float, longitude: float, when: pd.Timestamp, station_ids: Optional[Iterable[int]] = None) -> Dict[str, Any]:
    raw_features, meta = feature_row_for_coordinate(latitude, longitude, when, station_ids=station_ids, label=label)
    predictions = predict_aqi_from_raw(raw_features)
    nearest = meta["nearest_stations"].iloc[0]
    return {
        "role": role,
        "location": label,
        "eta": when,
        "latitude": latitude,
        "longitude": longitude,
        "predicted_aqi": predictions["Ensemble"],
        "aqi_category": aqi_category(predictions["Ensemble"]),
        "pm25": raw_features.get("pm25"),
        "pm10": raw_features.get("pm10"),
        "temperature_c": raw_features.get("ambient_temperature"),
        "humidity_pct": raw_features.get("relative_humidity"),
        "openweather_aqi_index": meta.get("openweather_aqi_index"),
        "nearest_historical_station": nearest["city"],
        "nearest_station_distance_km": nearest["distance_km"],
    }


selected_route = next(route for route in candidate_routes if route["route_name"] == best_route_name)
route_geometry = selected_route.get("geometry_points") or selected_route.get("points", [])
route_duration_hours = float(selected_route.get("duration_hours", 0.0))

route_city_candidates = station_profiles.copy()
route_city_candidates["distance_to_route_km"] = route_city_candidates.apply(
    lambda row: nearest_distance_to_route_km(row["latitude"], row["longitude"], route_geometry),
    axis=1,
)
route_city_candidates["source_distance_km"] = route_city_candidates.apply(
    lambda row: haversine_km(row["latitude"], row["longitude"], source_location["latitude"], source_location["longitude"]),
    axis=1,
)
route_city_candidates["destination_distance_km"] = route_city_candidates.apply(
    lambda row: haversine_km(row["latitude"], row["longitude"], destination_location["latitude"], destination_location["longitude"]),
    axis=1,
)
route_city_candidates["route_progress"] = route_city_candidates.apply(
    lambda row: route_progress_fraction(row["latitude"], row["longitude"], route_geometry),
    axis=1,
)

between_cities = route_city_candidates[
    (route_city_candidates["distance_to_route_km"] <= ROUTE_CITY_CORRIDOR_KM)
    & (route_city_candidates["source_distance_km"] >= MIN_ROUTE_CITY_DISTANCE_FROM_ENDPOINT_KM)
    & (route_city_candidates["destination_distance_km"] >= MIN_ROUTE_CITY_DISTANCE_FROM_ENDPOINT_KM)
].copy()

if between_cities.empty:
    between_cities = route_city_candidates.nsmallest(8, "distance_to_route_km").copy()

between_cities = between_cities.sort_values(["route_progress", "distance_to_route_km"]).reset_index(drop=True)
if MAX_MAJOR_ROUTE_CITIES is not None:
    between_cities = between_cities.head(int(MAX_MAJOR_ROUTE_CITIES))

report_rows = [
    route_aqi_report_row(
        label=f"Source: {source_location['query']}",
        role="Source",
        latitude=source_location["latitude"],
        longitude=source_location["longitude"],
        when=TRAVEL_DATETIME,
        station_ids=source_location.get("station_ids"),
    )
]

for _, city_row in between_cities.iterrows():
    city_eta = TRAVEL_DATETIME + pd.Timedelta(hours=route_duration_hours * float(city_row["route_progress"]))
    report_rows.append(
        route_aqi_report_row(
            label=readable_monitoring_city(city_row["city"], city_row["state"]),
            role="Major route city",
            latitude=float(city_row["latitude"]),
            longitude=float(city_row["longitude"]),
            when=city_eta,
            station_ids=[int(city_row["station_id"])],
        )
    )

report_rows.append(
    route_aqi_report_row(
        label=f"Destination: {destination_location['query']}",
        role="Destination",
        latitude=destination_location["latitude"],
        longitude=destination_location["longitude"],
        when=TRAVEL_DATETIME + pd.Timedelta(hours=route_duration_hours),
        station_ids=destination_location.get("station_ids"),
    )
)

source_destination_city_aqi_report = pd.DataFrame(report_rows)
print("AQI of source, destination, and major monitored cities between them")
display(source_destination_city_aqi_report)

AQI of source, destination, and major monitored cities between them


,role,location,eta,latitude,longitude,predicted_aqi,aqi_category,pm25,pm10,temperature_c,humidity_pct,openweather_aqi_index,nearest_historical_station,nearest_station_distance_km
0,Source,Source: new delhi,2026-07-14 20:00:00.000000000,28.613895,77.209006,247.498293,Poor,17.05,54.03,39.20,25.0,3,"Ashok Vihar (, )",9.446447
1,Major route city,"Alipur, Delhi",2026-07-14 20:00:00.000000000,28.797900,77.123300,173.597035,Moderate,17.98,60.50,37.69,30.0,3,"Alipur (, )",0.000000
2,Major route city,"Bawana, Delhi",2026-07-14 20:00:00.000000000,28.776200,77.051070,161.801856,Moderate,17.98,60.50,37.15,32.0,3,"Bawana (, )",0.000000
3,Major route city,"Bahadurgarh (Arya Nagar, Harayana)",2026-07-14 20:00:00.000000000,28.676400,76.929500,281.876165,Poor,18.91,66.97,37.33,33.0,3,"Arya Nagar, Bahadurgarh (, )",0.000000
4,Major route city,"Gurugram (Teri Gram, Harayana)",2026-07-14 20:21:46.763702604,28.428000,77.148000,283.525829,Poor,16.80,56.66,36.78,33.0,3,"Teri Gram, Gurugram (, )",0.000000
5,Major route city,"Faridabad (Sector 11, Harayana)",2026-07-14 20:25:15.435393334,28.374890,77.317970,282.928145,Poor,14.85,47.50,36.43,36.0,2,"Sector 11, Faridabad (, )",0.000000
6,Major route city,"Ballabgarh (Nathu Colony, Harayana)",2026-07-14 20:27:35.690136283,28.341600,77.320500,155.883335,Moderate,14.85,47.50,35.64,39.0,2,"Nathu Colony, Ballabgarh (, )",0.000000
7,Major route city,"Agra (Manoharpur, Uttar Pradesh)",2026-07-14 22:04:51.655781945,27.230800,78.027200,168.437559,Moderate,14.34,43.87,38.14,33.0,2,"Manoharpur, Agra (, )",0.000000
8,Major route city,"Firozabad (Nagla Bhau, Uttar Pradesh)",2026-07-14 22:20:59.755593037,27.159101,78.395760,78.376913,Satisfactory,17.06,48.07,37.94,32.0,2,"Nagla Bhau, Firozabad ( , )",0.000000
9,Major route city,"Gwalior (City Center, Madhya Pradesh)",2026-07-14 23:34:53.173809202,26.218287,78.182830,34.945534,Good,15.57,40.12,38.31,28.0,2,"City Center, Gwalior (, )",0.000000


In [13]:
# =============================================================================
# Live weather and pollution report for source and destination
# =============================================================================

required_objects = ["source_location", "destination_location"]
missing_objects = [name for name in required_objects if name not in globals()]
if missing_objects:
    raise RuntimeError(f"Run the end-to-end live inference cell first. Missing: {missing_objects}")


def openweather_aqi_label(index: Any) -> str:
    labels = {
        1: "Good",
        2: "Fair",
        3: "Moderate",
        4: "Poor",
        5: "Very Poor",
    }
    try:
        return labels.get(int(index), "Unknown")
    except (TypeError, ValueError):
        return "Unknown"


def live_weather_pollution_report_row(label: str, location: Dict[str, Any]) -> Dict[str, Any]:
    weather_values, weather_meta = fetch_current_weather(location["latitude"], location["longitude"])
    pollution_values, pollution_meta = fetch_current_air_pollution(location["latitude"], location["longitude"])
    return {
        "location": label,
        "resolved_name": location.get("display_name"),
        "latitude": location["latitude"],
        "longitude": location["longitude"],
        "weather": weather_meta.get("weather_description"),
        "temperature_c": weather_values.get("ambient_temperature"),
        "humidity_pct": weather_values.get("relative_humidity"),
        "rainfall_mm": weather_values.get("rainfall"),
        "openweather_aqi_index": pollution_meta.get("openweather_aqi_index"),
        "openweather_aqi_label": openweather_aqi_label(pollution_meta.get("openweather_aqi_index")),
        "pm25_ug_m3": pollution_values.get("pm25"),
        "pm10_ug_m3": pollution_values.get("pm10"),
        "co_mg_m3": pollution_values.get("carbon_monoxide"),
        "no_ug_m3": pollution_values.get("nitric_oxide"),
        "no2_ug_m3": pollution_values.get("nitrogen_dioxide"),
        "nox_ug_m3": pollution_values.get("nitrogen_oxides"),
        "so2_ug_m3": pollution_values.get("sulfur_dioxide"),
        "o3_ug_m3": pollution_values.get("ozone"),
        "nh3_ug_m3": pollution_values.get("ammonia"),
        "pollution_observed_at_utc": pollution_meta.get("air_pollution_timestamp"),
    }


source_destination_live_report = pd.DataFrame([
    live_weather_pollution_report_row(f"Source: {source_location['query']}", source_location),
    live_weather_pollution_report_row(f"Destination: {destination_location['query']}", destination_location),
])

print("Live weather and pollution report for source and destination")
display(source_destination_live_report)

Live weather and pollution report for source and destination


,location,resolved_name,latitude,longitude,weather,temperature_c,humidity_pct,rainfall_mm,openweather_aqi_index,openweather_aqi_label,pm25_ug_m3,pm10_ug_m3,co_mg_m3,no_ug_m3,no2_ug_m3,nox_ug_m3,so2_ug_m3,o3_ug_m3,nh3_ug_m3,pollution_observed_at_utc
0,Source: new delhi,"New Delhi, Delhi, IN",28.613895,77.209006,few clouds,39.20,25.0,0.0,3,Moderate,17.05,54.03,0.14488,0.02,5.66,5.68,1.55,75.29,5.54,2026-07-14 14:04:29
1,Destination: patna,"patna (historical match: GVM Corporation, Visa...",17.720000,83.300000,overcast clouds,25.08,84.0,0.0,2,Fair,20.11,31.10,0.16619,0.00,5.72,5.72,6.15,63.22,7.59,2026-07-14 14:13:25
